In [0]:
from pyspark.sql.functions import *

transactions_bronze = spark.read.table("my_catalog.bronze.transactions_data")
cards_bronze = spark.read.table("my_catalog.bronze.cards_data")
users_bronze = spark.read.table("my_catalog.bronze.users_data")

In [0]:
transactions_silver = (
    transactions_bronze
    .dropDuplicates()
    .withColumn("id", col("id").cast("string"))
    .withColumn("client_id", col("client_id").cast("string"))
    .withColumn("card_id", col("card_id").cast("string"))
    .withColumn("merchant_id", col("merchant_id").cast("string"))
    .withColumn("mcc", col("mcc").cast("string"))
    .withColumn("zip", col("zip").cast("string"))
    .withColumn("amount", regexp_replace(col("amount"), "[$,]", "").cast("double"))
    .withColumn("transaction_ts", to_timestamp(col("date")))
    .withColumn("transaction_date", to_date(col("transaction_ts")))
    .withColumn("transaction_hour", hour(col("transaction_ts")))
    .withColumn("day_of_week", date_format(col("transaction_ts"), "EEEE"))
    .withColumn("week_start", date_trunc("week", col("transaction_ts")))
    .withColumn("year_month", date_format(col("transaction_ts"), "yyyy-MM"))
    .withColumn(
        "time_of_day",
        when((col("transaction_hour") >= 5) & (col("transaction_hour") < 12), "morning")
        .when((col("transaction_hour") >= 12) & (col("transaction_hour") < 17), "afternoon")
        .when((col("transaction_hour") >= 17) & (col("transaction_hour") < 21), "evening")
        .otherwise("night")
    )
    .dropna(subset=["id", "client_id", "card_id", "amount", "transaction_ts", "mcc"])
)

display(transactions_silver.limit(5))

id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors,transaction_ts,transaction_date,transaction_hour,day_of_week,week_start,year_month,time_of_day
9807202,2011-07-16 22:16:00,901,4152,3.74,Swipe Transaction,20561,Pensacola,FL,32507.0,5912,null,2011-07-16T22:16:00.000Z,2011-07-16,22,Saturday,2011-07-11T00:00:00.000Z,2011-07,night
9807597,2011-07-17 06:12:00,236,4560,11.4,Swipe Transaction,24504,Katy,TX,77449.0,4214,null,2011-07-17T06:12:00.000Z,2011-07-17,6,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning
9808221,2011-07-17 08:56:00,1253,3841,53.0,Swipe Transaction,43293,Panama City,FL,32401.0,5499,null,2011-07-17T08:56:00.000Z,2011-07-17,8,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning
9808868,2011-07-17 11:22:00,1248,5787,11.01,Swipe Transaction,22544,Janesville,WI,53546.0,5912,null,2011-07-17T11:22:00.000Z,2011-07-17,11,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning
9809388,2011-07-17 13:22:00,1422,5515,2.43,Swipe Transaction,43293,Benson,NC,27504.0,5499,null,2011-07-17T13:22:00.000Z,2011-07-17,13,Sunday,2011-07-11T00:00:00.000Z,2011-07,afternoon


In [0]:
cards_silver = (
    cards_bronze
    .dropDuplicates()
    .withColumn("id", col("id").cast("string"))
    .withColumn("client_id", col("client_id").cast("string"))
    .withColumn("credit_limit", regexp_replace(col("credit_limit"), "[$,]", "").cast("double"))
    .withColumn("has_chip", col("has_chip").cast("string"))
    .withColumn("num_cards_issued", col("num_cards_issued").cast("int"))
    .withColumn("expires", col("expires").cast("string"))
    .withColumn("acct_open_date", col("acct_open_date").cast("string"))
)

display(cards_silver.limit(5))

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,1362,Amex,Credit,393314135668401,04/2024,866,YES,2,33900.0,01/1991,2014,No
1,550,Mastercard,Credit,5278231764792292,06/2024,396,YES,1,11600.0,01/1994,2013,No
2,556,Mastercard,Debit,5889825928297675,09/2021,422,YES,1,19948.0,01/1995,2011,No
3,1937,Visa,Credit,4289888672554714,04/2020,736,YES,2,16400.0,01/1995,2015,No
4,1981,Mastercard,Debit,5433366978583845,03/2024,530,YES,2,19439.0,01/1997,2007,No


In [0]:
users_silver = (
    users_bronze
    .dropDuplicates()
    .withColumn("id", col("id").cast("string"))
    .withColumn("current_age", col("current_age").cast("int"))
    .withColumn("retirement_age", col("retirement_age").cast("int"))
    .withColumn("birth_year", col("birth_year").cast("int"))
    .withColumn("birth_month", col("birth_month").cast("int"))
    .withColumn("latitude", col("latitude").cast("double"))
    .withColumn("longitude", col("longitude").cast("double"))
    .withColumn("per_capita_income", regexp_replace(col("per_capita_income"), "[$,]", "").cast("double"))
    .withColumn("yearly_income", regexp_replace(col("yearly_income"), "[$,]", "").cast("double"))
    .withColumn("total_debt", regexp_replace(col("total_debt"), "[$,]", "").cast("double"))
    .withColumn("credit_score", col("credit_score").cast("int"))
    .withColumn("num_credit_cards", col("num_credit_cards").cast("int"))
)

display(users_silver.limit(5))

id,current_age,retirement_age,birth_year,birth_month,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [0]:
import json

mcc_raw_df = (
    spark.read
    .option("wholetext", "true")
    .text("/Volumes/jarvis_etl/default/json/mcc_codes.json")
)

raw_text = mcc_raw_df.selectExpr("collect_list(value)").first()[0]
full_json = " ".join(raw_text)

mcc_lookup = (
    spark.createDataFrame(
        list(json.loads(full_json).items()),
        ["mcc", "merchant_category"]
    )
)

display(mcc_lookup.limit(10))

mcc,merchant_category
5812,Eating Places and Restaurants
5541,Service Stations
7996,"Amusement Parks, Carnivals, Circuses"
5411,"Grocery Stores, Supermarkets"
4784,Tolls and Bridge Fees
4900,"Utilities - Electric, Gas, Water, Sanitary"
5942,Book Stores
5814,Fast Food Restaurants
4829,Money Transfer
5311,Department Stores


In [0]:
import json

fraud_raw_df = (
    spark.read
    .option("wholetext", "true")
    .text("/Volumes/jarvis_etl/default/json/train_fraud_labels.json")
)

raw_text = fraud_raw_df.selectExpr("collect_list(value)").first()[0]
full_json = " ".join(raw_text)

parsed = json.loads(full_json)

target_map = parsed.get("target", parsed)  

fraud_lookup = (
    spark.createDataFrame(
        list(target_map.items()),
        ["id", "is_fraud"]
    )
)

display(fraud_lookup.limit(10))

id,is_fraud
10649266,No
23410063,No
9316588,No
12478022,No
9558530,No
12532830,No
19526714,No
9906964,No
13224888,No
13749094,No


In [0]:
import json
from pyspark.sql.functions import when

# Step 1: Enhance with fraud labels and MCC
transactions_enhanced = (
    transactions_silver
    .join(
        fraud_lookup.select("id", "is_fraud"),
        on="id",
        how="left"
    )
    .join(
        mcc_lookup.select("mcc", "merchant_category"),
        on="mcc",
        how="left"
    )
    .withColumn(
        "is_fraud",
        when(col("is_fraud") == "Yes", 1)
        .when(col("is_fraud") == "No", 0)
        .otherwise(0)
        .cast("integer")
    )
)
display(transactions_enhanced.limit(10))

mcc,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_ts,transaction_date,transaction_hour,day_of_week,week_start,year_month,time_of_day,is_fraud,merchant_category
4829,9809596,2011-07-17 14:06:00,1105,2036,40.0,Swipe Transaction,27092,Leander,TX,78641.0,null,2011-07-17T14:06:00.000Z,2011-07-17,14,Sunday,2011-07-11T00:00:00.000Z,2011-07,afternoon,0,Money Transfer
5499,9812021,2011-07-18 07:25:00,1288,1013,0.84,Swipe Transaction,59935,Saint Albans,WV,25177.0,null,2011-07-18T07:25:00.000Z,2011-07-18,7,Monday,2011-07-18T00:00:00.000Z,2011-07,morning,0,Miscellaneous Food Stores
5411,9807944,2011-07-17 07:44:00,1022,5148,90.64,Swipe Transaction,98374,Mooresboro,NC,28114.0,null,2011-07-17T07:44:00.000Z,2011-07-17,7,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning,0,"Grocery Stores, Supermarkets"
5541,9808767,2011-07-17 11:01:00,547,5526,13.88,Swipe Transaction,72351,Delavan,WI,53115.0,null,2011-07-17T11:01:00.000Z,2011-07-17,11,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning,0,Service Stations
3771,9810577,2011-07-17 18:16:00,1769,5599,95.1,Swipe Transaction,39991,Lincoln Park,MI,48146.0,null,2011-07-17T18:16:00.000Z,2011-07-17,18,Sunday,2011-07-11T00:00:00.000Z,2011-07,evening,0,Railroad Passenger Transport
5942,9812857,2011-07-18 10:06:00,1069,5167,2.11,Swipe Transaction,20519,Charlotte,NC,28216.0,null,2011-07-18T10:06:00.000Z,2011-07-18,10,Monday,2011-07-18T00:00:00.000Z,2011-07,morning,0,Book Stores
4829,9813432,2011-07-18 11:59:00,938,231,100.0,Swipe Transaction,27092,Lawrence,KS,66049.0,null,2011-07-18T11:59:00.000Z,2011-07-18,11,Monday,2011-07-18T00:00:00.000Z,2011-07,morning,0,Money Transfer
4900,9811343,2011-07-17 23:28:00,146,1143,165.19,Swipe Transaction,57649,Batavia,IL,60510.0,null,2011-07-17T23:28:00.000Z,2011-07-17,23,Sunday,2011-07-11T00:00:00.000Z,2011-07,night,0,"Utilities - Electric, Gas, Water, Sanitary"
5499,9817892,2011-07-19 12:19:00,1459,3383,3.4,Swipe Transaction,43293,Albuquerque,NM,87121.0,null,2011-07-19T12:19:00.000Z,2011-07-19,12,Tuesday,2011-07-18T00:00:00.000Z,2011-07,afternoon,0,Miscellaneous Food Stores
5541,9810896,2011-07-17 20:11:00,1008,2592,3.83,Swipe Transaction,61195,Chicago,IL,60630.0,null,2011-07-17T20:11:00.000Z,2011-07-17,20,Sunday,2011-07-11T00:00:00.000Z,2011-07,evening,0,Service Stations


In [0]:
# Step 2: Enrich with card and user data
tx = transactions_enhanced

transactions_enhanced = (
    tx
    .join(
        cards_silver.select(
            col("id").alias("card_join_id"),
            "card_brand", "card_type", "has_chip", "credit_limit", "card_on_dark_web"
        ),
        tx["card_id"] == col("card_join_id"),
        "left"
    )
    .drop("card_join_id")
)

tx2 = transactions_enhanced

transactions_enhanced = (
    tx2
    .join(
        users_silver.select(
            col("id").alias("user_join_id"),
            "current_age", "retirement_age", "gender", "address",
            "latitude", "longitude", "per_capita_income", "yearly_income",
            "total_debt", "credit_score", "num_credit_cards"
        ),
        tx2["client_id"] == col("user_join_id"),
        "left"
    )
    .drop("user_join_id")
)

display(transactions_enhanced.limit(10))

mcc,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,errors,transaction_ts,transaction_date,transaction_hour,day_of_week,week_start,year_month,time_of_day,is_fraud,merchant_category,card_brand,card_type,has_chip,credit_limit,card_on_dark_web,current_age,retirement_age,gender,address,latitude,longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
4829,9809596,2011-07-17 14:06:00,1105,2036,40.0,Swipe Transaction,27092,Leander,TX,78641.0,null,2011-07-17T14:06:00.000Z,2011-07-17,14,Sunday,2011-07-11T00:00:00.000Z,2011-07,afternoon,0,Money Transfer,Mastercard,Debit,YES,9027.0,No,77,60,Male,766 Mountain View Drive,26.21,-98.31,14275.0,22104.0,20031.0,670,3
5499,9812021,2011-07-18 07:25:00,1288,1013,0.84,Swipe Transaction,59935,Saint Albans,WV,25177.0,null,2011-07-18T07:25:00.000Z,2011-07-18,7,Monday,2011-07-18T00:00:00.000Z,2011-07,morning,0,Miscellaneous Food Stores,Mastercard,Debit,YES,9796.0,No,41,57,Female,14 Valley Drive,38.47,-81.81,18666.0,38059.0,68220.0,634,2
5411,9807944,2011-07-17 07:44:00,1022,5148,90.64,Swipe Transaction,98374,Mooresboro,NC,28114.0,null,2011-07-17T07:44:00.000Z,2011-07-17,7,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning,0,"Grocery Stores, Supermarkets",Mastercard,Debit,YES,16274.0,No,47,64,Female,27 Summit Street,34.98,-80.54,17595.0,35876.0,50190.0,635,3
5541,9808767,2011-07-17 11:01:00,547,5526,13.88,Swipe Transaction,72351,Delavan,WI,53115.0,null,2011-07-17T11:01:00.000Z,2011-07-17,11,Sunday,2011-07-11T00:00:00.000Z,2011-07,morning,0,Service Stations,Mastercard,Debit,NO,20998.0,No,70,66,Male,111 Hillside Avenue,42.62,-88.63,18351.0,31901.0,16285.0,778,4
3771,9810577,2011-07-17 18:16:00,1769,5599,95.1,Swipe Transaction,39991,Lincoln Park,MI,48146.0,null,2011-07-17T18:16:00.000Z,2011-07-17,18,Sunday,2011-07-11T00:00:00.000Z,2011-07,evening,0,Railroad Passenger Transport,Mastercard,Debit,YES,17335.0,No,75,65,Male,177 Mill Boulevard,39.69,-74.25,21861.0,46031.0,8332.0,661,5
5942,9812857,2011-07-18 10:06:00,1069,5167,2.11,Swipe Transaction,20519,Charlotte,NC,28216.0,null,2011-07-18T10:06:00.000Z,2011-07-18,10,Monday,2011-07-18T00:00:00.000Z,2011-07,morning,0,Book Stores,Visa,Debit,YES,2292.0,No,54,69,Male,3683 Fifth Street,35.3,-81.03,22487.0,45852.0,114746.0,615,2
4829,9813432,2011-07-18 11:59:00,938,231,100.0,Swipe Transaction,27092,Lawrence,KS,66049.0,null,2011-07-18T11:59:00.000Z,2011-07-18,11,Monday,2011-07-18T00:00:00.000Z,2011-07,morning,0,Money Transfer,Mastercard,Credit,YES,12500.0,No,28,73,Female,875 Plum Street,38.97,-94.95,24982.0,50934.0,54406.0,720,3
4900,9811343,2011-07-17 23:28:00,146,1143,165.19,Swipe Transaction,57649,Batavia,IL,60510.0,null,2011-07-17T23:28:00.000Z,2011-07-17,23,Sunday,2011-07-11T00:00:00.000Z,2011-07,night,0,"Utilities - Electric, Gas, Water, Sanitary",Mastercard,Credit,YES,14200.0,No,47,67,Female,45 Ocean Lane,41.85,-88.3,33701.0,68710.0,201796.0,717,3
5499,9817892,2011-07-19 12:19:00,1459,3383,3.4,Swipe Transaction,43293,Albuquerque,NM,87121.0,null,2011-07-19T12:19:00.000Z,2011-07-19,12,Tuesday,2011-07-18T00:00:00.000Z,2011-07,afternoon,0,Miscellaneous Food Stores,Mastercard,Debit,NO,12708.0,No,53,65,Male,9385 Birch Street,35.11,-106.62,15385.0,31369.0,0.0,707,3
5541,9810896,2011-07-17 20:11:00,1008,2592,3.83,Swipe Transaction,61195,Chicago,IL,60630.0,null,2011-07-17T20:11:00.000Z,2011-07-17,20,Sunday,2011-07-11T00:00:00.000Z,2011-07,evening,0,Service Stations,Visa,Debit,YES,20245.0,No,52,69,Female,38688 First Avenue,41.83,-87.68,22296.0,45462.0,68494.0,657,5


In [0]:
# Step 3: Write silver tables
transactions_enhanced.write.format("delta").mode("overwrite").saveAsTable("my_catalog.silver.transactions")
cards_silver.write.format("delta").mode("overwrite").saveAsTable("my_catalog.silver.cards")
users_silver.write.format("delta").mode("overwrite").saveAsTable("my_catalog.silver.users")
mcc_lookup.write.format("delta").mode("overwrite").saveAsTable("my_catalog.silver.mcc_codes")
fraud_lookup.write.format("delta").mode("overwrite").saveAsTable("my_catalog.silver.fraud_labels")